# DATAANALYSIS — Vietnam's 2025 administrative-merger atlas

Deep analytical walkthrough of the curated parquet bundle produced by
`geography-vn curate`. We load two artefacts:

* `data/sapnhap-bando-vn/extracted/extracted.parquet` — every entity with the
  derived analytical columns (macro-region, predecessors list, keywords).
* `data/sapnhap-bando-vn/reduced/reduced.parquet` — the same rows + UMAP
  2-D coordinates and HDBSCAN cluster ids.

Two figure packs ship side-by-side:

* **Analytical** — bar charts, distributions, decree corpus, curator UMAP.
  Rendered by `python -m scripts.analyze` to `docs/figures/analysis/` and
  reproduced inline below. LaTeX-serif typography on a white canvas.
* **Cartographic** — choropleths + commune / committee scatter maps with
  the dual-archipelago declaration (Hoàng Sa + Trường Sa). Rendered
  standalone by `python -m scripts.render_maps` to `docs/figures/maps/`
  because kaleido + Mapbox-GL crashes inside a Jupyter kernel. See
  [Section 5](#5-cartographic-pack) below for the inventory.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import pandas as pd
from packages.curator.regions import MACRO_REGION_EN
from packages.viz.style import register_plotly_template
from scripts.analyze import (
    fig_kind_donut, fig_macro_region,
    fig_province_metric, fig_communes_per_province,
    fig_merger_fanout, fig_commune_size_distribution,
    fig_decree_map, fig_curator_umap,
)

register_plotly_template('nvidia_latex')

ROOT = REPO / 'data' / 'sapnhap-bando-vn'
df = pd.read_parquet(ROOT / 'extracted' / 'extracted.parquet')
rdf = pd.read_parquet(ROOT / 'reduced' / 'reduced.parquet')
print(f'extracted: {len(df):,} rows; reduced: {len(rdf):,} rows')
print(df['kind'].value_counts().to_string())

extracted: 6,712 rows; reduced: 6,712 rows
kind
committee    3357
commune      3321
province       34


## 1 — Inventory

In [2]:
fig_kind_donut(df).show()

In [3]:
fig_macro_region(df).show()

## 2 — Province-level summaries

In [4]:
fig_province_metric(df, metric='population', unit='people',
                    title='34 provinces by population').show()

In [5]:
fig_province_metric(df, metric='area_km2', unit='km²',
                    title='34 provinces by area').show()

In [6]:
fig_province_metric(df, metric='density', unit='people / km²',
                    title='34 provinces by population density').show()

In [7]:
fig_communes_per_province(df).show()

## 3 — Merger fanout

How many predecessor units (provinces, wards, xã) feed each surviving
first-level and second-level admin unit? `n_predecessors=1` means the
unit kept its borders; ≥2 means it absorbed neighbours.

In [8]:
fig_merger_fanout(df, kind='province',
                  title='Province merger fanout').show()

In [9]:
fig_merger_fanout(df, kind='commune',
                  title='Commune merger fanout').show()

## 4 — Distributions

In [10]:
fig_commune_size_distribution(df).show()

In [11]:
fig_decree_map(df).show()

## 5 — Cartographic pack

Five geographic figures with the **dual-archipelago declaration**
(Quần đảo Hoàng Sa + Quần đảo Trường Sa) live under `docs/figures/maps/`,
rendered standalone by `python -m scripts.render_maps`:

| File                                  | What it shows                                                |
| ------------------------------------- | ------------------------------------------------------------ |
| `01_provinces_population.png`         | Population by post-merger province (NVIDIA-green choropleth) |
| `02_provinces_density.png`            | People per km² (sequential)                                  |
| `03_provinces_area.png`               | Land area (km², sequential)                                  |
| `04_communes_scatter.png`             | 3,321 commune centroids, coloured by macro-region            |
| `05_committees_scatter.png`           | 3,357 commune people's-committee headquarters by lat/lon     |

Each map carries: ★ Thủ đô Hà Nội, 7 named cities (TP. Hồ Chí Minh /
Đà Nẵng / Hải Phòng / Huế / Vinh / Cần Thơ / Nha Trang) with leader-line
labels, 5 island callouts (Đảo Phú Quốc / Cát Bà / Bạch Long Vĩ /
Lý Sơn + Côn Đảo), and bilingual archipelago labels.

## 6 — Curator UMAP

The 384-d sentence-transformers vectors of the merger-lineage descriptors,
projected to 2-D with UMAP (cosine, 15 neighbours, `min_dist=0.1`).

In [12]:
fig_curator_umap(rdf, by='kind').show()

In [13]:
fig_curator_umap(rdf, by='macro_region').show()

## 7 — Quantitative summaries

Headline figures the dataset card quotes.

In [14]:
provinces = df[df['kind'] == 'province']
communes  = df[df['kind'] == 'commune']
print(f'Provinces: {len(provinces)}')
print(f'Communes:  {len(communes):,}')
print(f'Population total (provinces sum): {provinces["population"].dropna().sum():,.0f}')
print(f'Land area total (provinces sum): {provinces["area_km2"].dropna().sum():,.2f} km²')
print()
print('Top 5 most populous provinces:')
print(provinces.sort_values('population', ascending=False).head(5)[['ten','population','area_km2','density']].to_string(index=False))
print()
print('Top 5 largest provinces by area:')
print(provinces.sort_values('area_km2', ascending=False).head(5)[['ten','area_km2','population','density']].to_string(index=False))
print()
print('Top 5 densest provinces:')
print(provinces.sort_values('density', ascending=False).head(5)[['ten','density','population','area_km2']].to_string(index=False))

Provinces: 34
Communes:  3,321
Population total (provinces sum): 113,571,926
Land area total (provinces sum): 331,325.62 km²

Top 5 most populous provinces:
                  ten  population  area_km2     density
Thành phố Hồ Chí Minh  14002598.0   6772.59 2067.539597
        Thủ đô Hà Nội   8807523.0   3359.84 2621.411436
        Tỉnh An Giang   4952238.0   9888.91  500.787043
  Thành phố Hải Phòng   4664124.0   3194.72 1459.947664
   Thành phố Đồng Nai   4491408.0  12737.18  352.621852

Top 5 largest provinces by area:
            ten  area_km2  population    density
  Tỉnh Lâm Đồng  24233.07   3872999.0 159.822878
   Tỉnh Gia Lai  21576.53   3583693.0 166.092184
   Tỉnh Đắk Lắk  18096.40   3346853.0 184.945790
   Tỉnh Nghệ An  16486.50   3831694.0 232.414036
Tỉnh Quảng Ngãi  14832.55   2161755.0 145.743989

Top 5 densest provinces:
                  ten     density  population  area_km2
        Thủ đô Hà Nội 2621.411436   8807523.0   3359.84
Thành phố Hồ Chí Minh 2067.539597  140025

In [15]:
print('Commune merger-fanout summary:')
fan = communes['n_predecessors'].fillna(0).astype(int)
for k, v in fan.value_counts().sort_index().items():
    print(f'  {k:>2} predecessor(s): {v:>5,d} communes ({v/len(communes):.1%})')
print()
print(f'Mean fanout:   {fan.mean():.2f}')
print(f'Median fanout: {fan.median():.0f}')
print(f'Max fanout:    {fan.max()}')

Commune merger-fanout summary:
   1 predecessor(s):   139 communes (4.2%)
   2 predecessor(s):   758 communes (22.8%)
   3 predecessor(s): 1,498 communes (45.1%)
   4 predecessor(s):   559 communes (16.8%)
   5 predecessor(s):   179 communes (5.4%)
   6 predecessor(s):    78 communes (2.3%)
   7 predecessor(s):    37 communes (1.1%)
   8 predecessor(s):    29 communes (0.9%)
   9 predecessor(s):    17 communes (0.5%)
  10 predecessor(s):    12 communes (0.4%)
  11 predecessor(s):     6 communes (0.2%)
  12 predecessor(s):     6 communes (0.2%)
  13 predecessor(s):     1 communes (0.0%)
  14 predecessor(s):     1 communes (0.0%)
  16 predecessor(s):     1 communes (0.0%)

Mean fanout:   3.22
Median fanout: 3
Max fanout:    16
